# 🛰️ BPS Run — MAAP Coding Environment 🛰️
Copyright 2025, European Space Agency (ESA) Licensed under ESA Software Community Licence Permissive (Type 3) - v2.4

**Procedure for running the BPS Processor Suite (BPS) within the MAAP Coding/Experiment environment.**

This notebook guides through the full BPS processing chain, from JobOrder generation up to the L2A output products.

---

## 📋 Overview

| Step | Description |
|---|---|
| **0. Configuration** | Define BPS version, input folder and all relevant paths |
| **1. Check environment** | Verify that the `biofetch` conda environment exists |
| **2. Check input folder** | Verify that input products are present in `INPUT_FOLDER/Inputs/` |
| **3. L1F chain** | Generate L1F JobOrders and run `bps_l1_framing_processor` |
| **4. L1 chain** | Generate L1 JobOrders and run `bps_l1_processor` |
| **5. STA chain** | Generate STA JobOrders and run `bps_stack_processor` |
| **6. L2A chain** | Generate L2A JobOrders and run `bps_l2a_processor` |

---

## 📦 Input products

Place all input products in `INPUT_FOLDER/Inputs/` before running the notebook:

| Product type | Description |
|---|---|
| `BIO_S1_RAW__0S_*` | Raw Single Look Complex — one per acquisition |
| `BIO_S1_RAW__0M_*` | Raw Monitoring — one per acquisition |
| `BIO_AUX_ORB_*` | Orbit auxiliary data — one per acquisition |
| `BIO_AUX_ATT_*` | Attitude auxiliary data — one per acquisition |
| `BIO_AUX_TEC_*` | TEC auxiliary data — one per day, shared across acquisitions |

Expected folder structure:
```
INPUT_FOLDER/Inputs/
├── BIO_AUX_ATT____20251121T094455_20251121T095757_01_DIGNJT
├── BIO_AUX_ATT____20251124T094456_20251124T095759_01_DIM8IK
├── BIO_AUX_ORB____20251121T094455_20251121T095757_01_DIGNJT
├── BIO_AUX_ORB____20251124T094456_20251124T095759_01_DIM8IK
├── BIO_AUX_TEC____20251121T000000_20251121T235959_01_DIFMBZ
├── BIO_AUX_TEC____20251124T000000_20251124T235959_01_DIL6HR
├── BIO_S1_RAW__0M_20251121T094510_20251121T095758_T_G01_M01_C01_T006_F____01_DJUQ12
├── BIO_S1_RAW__0M_20251124T094512_20251124T095800_T_G01_M01_C02_T006_F____01_DK5M74
├── BIO_S1_RAW__0S_20251121T095253_20251121T095450_T_G01_M01_C01_T006_F062_01_DNG988
└── BIO_S1_RAW__0S_20251124T095255_20251124T095452_T_G01_M01_C02_T006_F062_01_DK5GET
```

> ℹ️ Each acquisition contributes one RAW0S, one RAW0M, one AUX_ORB and one AUX_ATT.  
> AUX_TEC covers a full day and is shared across acquisitions on the same date.
----
## 🗂️ Auxiliary products for L1

The L1 processor requires additional AUX files that are placed in in a dedicated AUX directory. The path is configured in `config.ini`:

```ini
[AUX]
AUX_DEFAULT_DIR = /home/jovyan/SCRIPTS/CONFIGURATION_FILE/AUX_441
AUX_USER_DIR    = /home/jovyan/SCRIPTS/CONFIGURATION_FILE/AUX_USER
```

`AUX_DEFAULT_DIR` is the default location. `AUX_USER_DIR` can be used to override individual AUX files — files found there take priority over `AUX_DEFAULT_DIR`.

Expected contents of `AUX_DEFAULT_DIR`:

```
AUX_441/
├── BIO_AUX_INS____20250522T000000_20260131T000000_01_DIE6O0
├── BIO_AUX_PP1____20250522T000000_20260131T000000_01_DIE6O0
├── BIO_AUX_PP2_2A_20170101T000000_20991231T000000_01_BHDHC0
├── BIO_AUX_PP2_AB_20170101T000000_20991231T000000_01_BHDHC0
├── BIO_AUX_PP2_FD_20170101T000000_20991231T000000_01_BHDHC0
├── BIO_AUX_PP2_FH_20170101T000000_20991231T000000_01_BHDHC0
└── BIO_AUX_PPS____20250522T000000_20260131T000000_01_DIE6O0
```

| Product type | Description |
|---|---|
| `BIO_AUX_INS_*` | Instrument characterisation auxiliary data |
| `BIO_AUX_PP1_*` | Level-1 processing parameters |
| `BIO_AUX_PPS_*` | Processing parameters for stack processor |
| `BIO_AUX_PP2_2A_*` | Level-2A processing parameters |
| `BIO_AUX_PP2_AB_*` | Level-2 AGB processing parameters |
| `BIO_AUX_PP2_FD_*` | Level-2 FD processing parameters |
| `BIO_AUX_PP2_FH_*` | Level-2 FH processing parameters |


> ℹ️ The AUX path is resolved by `JOBuilder.py` reading `config.ini`. Make sure the paths in `config.ini` point to existing folders before running the notebook.
---

## 🔄 Processing chain

```
RAW0S + RAW0M + AUX_ORB + AUX_ATT + AUX_TEC
        │
        ▼
   L1F chain  →  SLC framed products
        │
        ▼
   L1 chain   →  L1 products
        │
        ▼
   STA chain  →  Stack products
        │
        ▼
   L2A chain  →  L2A products
```

---

> ⚠️ **Prerequisites**
> - BPS V4.4.x installed — see `BPS_installation.ipynb`
> - Input products placed in `INPUT_FOLDER/Inputs/` as described above
> - `JOBuilder.py` and `biofetch.yml` available in `~/SCRIPTS/`


## 0. ⚙️ Configuration

Define all the variables used throughout the notebook.  
Edit `INPUT_FOLDER` and `PROCESSOR_VERSION` here — all paths are derived automatically.

| Variable | Description |
|---|---|
| `INPUT_FOLDER` | Folder containing the `Inputs/` subfolder with the input products |
| `PROCESSOR_VERSION` | BPS version to use (e.g. `04.41`) |
| `BPS_ENV` | Conda environment of the BPS processors (e.g. `BPS_441`) |
| `JO_ENV` | Conda environment used to run `JOBuilder.py` (`biofetch`) |
| `CONFIGURATION_BPS_DIR` | BTK deployment directory (e.g. `/home/jovyan/BPS/BPS_V441`) |
| `L1_FRAMES_FILTER` | Space-separated frame IDs to process (leave empty for all frames) |
| `MISSION_PHASE` | Mission phase for the STA chain: `INTERFEROMETRIC` or `TOMOGRAPHIC` |

In [1]:
# ============================================================
# CONFIGURATION — edit here before running any section
# ============================================================

HOME            = "/home/jovyan"
SCRIPTS_DIR     = f"{HOME}/SCRIPTS"
BIOFETCH_YML    = f"{SCRIPTS_DIR}/biofetch.yml"
JO_ENV       = "biofetch"

JOBUILDER       = f"{SCRIPTS_DIR}/JOBuilder.py"

# Input folder containing the products
INPUT_FOLDER    = f"{HOME}/DATA/01_manaus"   # ← change here

# Processor version
PROCESSOR_VERSION = "04.42"
CONFIGURATION_BPS_DIR = f"{HOME}/BPS/BPS_V{PROCESSOR_VERSION.replace('.', '').replace('0','')}"
BPS_ENV = f"BPS_{PROCESSOR_VERSION.replace('.', '').replace('0','')}"   # es. BPS_441
# Leave empty to process ALL frames, or specify frame IDs
L1_FRAMES_FILTER = "306"        # ← all frames
# L1_FRAMES_FILTER = "001 003"  # ← only F001 and F003

# Mission phase (for STA / STA_chain)
MISSION_PHASE   = "TOMOGRAPHIC"   # INTERFEROMETRIC or TOMOGRAPHIC

print("Configuration:")
print(f"  INPUT_FOLDER      : {INPUT_FOLDER}")
print(f"  PROCESSOR_VERSION : {PROCESSOR_VERSION}")
print(f"  BPS_ENV : {BPS_ENV}")
print(f"  CONFIGURATION_BPS_DIR : {CONFIGURATION_BPS_DIR}")
print(f"  MISSION_PHASE     : {MISSION_PHASE}")
print(f"  ENV_JO         : {JO_ENV}")

Configuration:
  INPUT_FOLDER      : /home/jovyan/DATA/01_manaus
  PROCESSOR_VERSION : 04.42
  BPS_ENV : BPS_442
  CONFIGURATION_BPS_DIR : /home/jovyan/BPS/BPS_V442
  MISSION_PHASE     : TOMOGRAPHIC
  ENV_JO         : biofetch


## 1. 🐍 Check Environment

Verifies that the `biofetch` conda environment exists.  
If not found, creates it from `biofetch.yml`.

In [2]:
%%bash -s "$JO_ENV" "$BIOFETCH_YML"
JO_ENV=$1
BIOFETCH_YML=$2

if conda env list | grep -qE "(^|[[:space:]])${JO_ENV}([[:space:]]|$)"; then
    echo "ℹ️  Environment ${JO_ENV} already exists, skipping installation"
else
    echo "📦 Creating environment ${JO_ENV} from ${BIOFETCH_YML} ..."
    conda env create -f "${BIOFETCH_YML}" --name "${JO_ENV}"
    echo "✅ Environment ${JO_ENV} created"
fi

echo ""
echo "=== Conda environments ==="
conda env list

ℹ️  Environment biofetch already exists, skipping installation

=== Conda environments ===

# conda environments:
#
BPS_441                /home/jovyan/env/BPS_441
BPS_442                /home/jovyan/env/BPS_442
biofetch               /home/jovyan/env/biofetch
base                 * /opt/conda



## 2. 📂 Check Input Folder

Verifies that `INPUT_FOLDER` exists and that the `Inputs/` subfolder is present and non-empty.

In [3]:
import os

required = ["Inputs"]

print(f"📂 Input folder: {INPUT_FOLDER}")
if not os.path.isdir(INPUT_FOLDER):
    print(f"❌ Input folder not found: {INPUT_FOLDER}")
else:
    for sub in required:
        path = os.path.join(INPUT_FOLDER, sub)
        if os.path.isdir(path):
            files = os.listdir(path)
            print(f"  ✅ {sub}/ found ({len(files)} file(s))")
        else:
            print(f"  ⚠️  {sub}/ not found")

📂 Input folder: /home/jovyan/DATA/01_manaus
  ✅ Inputs/ found (10 file(s))


## 3. 🛰️ L1F Chain

Generates the L1F JobOrders and runs `bps_l1_framing_processor` on each of them.

- **Step 1** — runs `JOBuilder.py L1F` (env: `biofetch`) to generate the L1F JobOrder XML files
- **Step 2** — runs `bps_l1_framing_processor` (env: `BPS_441`) on each JobOrder

In [4]:
%%bash -s "$JO_ENV" "$JOBUILDER" "$PROCESSOR_VERSION" "$INPUT_FOLDER" "$CONFIGURATION_BPS_DIR" "$BPS_ENV"
JO_ENV=$1
JOBUILDER=$2
PROCESSOR_VERSION=$3
INPUT_FOLDER=$4
CONFIGURATION_BPS_DIR=$5
BPS_ENV=$6

SCRIPTS_DIR="${CONFIGURATION_BPS_DIR}/tools/scripts"
source "${SCRIPTS_DIR}/set_environment.bash"
echo "✅ BPS environment sourced"

# ========================================
# Step 1/2 : Create JobOrders L1F
# ========================================
echo ""
echo "========================================"
echo "▶ Step 1/2 : Create JobOrders L1F"
echo "========================================"
conda run --name "${JO_ENV}" python "${JOBUILDER}" L1F "${PROCESSOR_VERSION}" "${INPUT_FOLDER}"
if [ $? -ne 0 ]; then echo "❌ JOBuilder L1F failed, stopping"; exit 1; fi

# ========================================
# Step 2/2 : Run bps_l1_framing_processor for each L1F JobOrder
# ========================================
echo ""
echo "========================================"
echo "▶ Step 2/2 : Run L1F processor"
echo "========================================"
JO_L1F_FILES=$(ls "${INPUT_FOLDER}"/JobOrder_*_L1F_*.xml 2>/dev/null)
if [ -z "${JO_L1F_FILES}" ]; then
    echo "❌ No L1F JobOrder files found in ${INPUT_FOLDER}"
    exit 1
fi

for JO in ${JO_L1F_FILES}; do
    echo ""
    echo "--- Running L1F processor on: $(basename ${JO}) ---"
    conda run --name "${BPS_ENV}" bps_l1_framing_processor "${JO}"
    if [ $? -ne 0 ]; then
        echo "❌ L1F processor failed on: ${JO}"
        exit 1
    fi
    echo "✅ Done: $(basename ${JO})"
done

echo ""
echo "✅ L1F completed"

✅ BPS environment sourced

▶ Step 1/2 : Create JobOrders L1F
createJOL1F
/home/jovyan/DATA/01_manaus/Inputs
20251121T09525 20251121T095450
20251124T09451 20251124T095800
20251121T09451 20251121T095758
20251121T09525 20251121T095450
['BIO_AUX_ORB____20251121T094455_20251121T095757_01_DIGNJT', 'BIO_AUX_ORB____20251124T094456_20251124T095759_01_DIM8IK']
20251121T09445 20251121T095757
2025-11-21 09:44:05 2025-11-21 09:57:57
09:44:05 09:57:57
09:52:05 09:54:50
True
/home/jovyan/DATA/01_manaus/Inputs/BIO_AUX_ORB____20251121T094455_20251121T095757_01_DIGNJT
{'RAW__0S': '/home/jovyan/DATA/01_manaus/Inputs/BIO_S1_RAW__0S_20251121T095253_20251121T095450_T_G01_M01_C01_T006_F062_01_DNG988', 'RAW__0M': '/home/jovyan/DATA/01_manaus/Inputs/BIO_S1_RAW__0M_20251121T094510_20251121T095758_T_G01_M01_C01_T006_F____01_DJUQ12', 'AUX_ORB': '/home/jovyan/DATA/01_manaus/Inputs/BIO_AUX_ORB____20251121T094455_20251121T095757_01_DIGNJT'}
__init__
2025-11-21 09:52:52.795000
2025-11-21 09:54:49.728000
/home/jovyan/

## 4. 📡 L1 Chain

Generates the L1 JobOrders and runs `bps_l1_processor` on the selected frames.

- **Step 1** — runs `JOBuilder.py L1_chain` (env: `biofetch`) to generate the L1 JobOrder XML files
- **Step 2** — runs `bps_l1_processor` (env: `BPS_441`) on each JobOrder, optionally filtered by frame ID via `L1_FRAMES_FILTER`

In [5]:
%%bash -s "$JO_ENV" "$JOBUILDER" "$PROCESSOR_VERSION" "$INPUT_FOLDER" "$CONFIGURATION_BPS_DIR" "$BPS_ENV" "$L1_FRAMES_FILTER"
JO_ENV=$1
JOBUILDER=$2
PROCESSOR_VERSION=$3
INPUT_FOLDER=$4
CONFIGURATION_BPS_DIR=$5
BPS_ENV=$6
L1_FRAMES_FILTER=$7   # space-separated filenames, or empty

SCRIPTS_DIR="${CONFIGURATION_BPS_DIR}/tools/scripts"
source "${SCRIPTS_DIR}/set_environment.bash"
echo "✅ BPS environment sourced"

# ========================================
# Step 1/2 : Create JobOrders L1_chain
# ========================================
echo ""
echo "========================================"
echo "▶ Step 1/2 : Create JobOrders L1_chain"
echo "========================================"
conda run --name "${JO_ENV}" python "${JOBUILDER}" L1_chain "${PROCESSOR_VERSION}" "${INPUT_FOLDER}"
if [ $? -ne 0 ]; then echo "❌ JOBuilder L1_chain failed, stopping"; exit 1; fi

# ========================================
# Step 2/2 : Run bps_l1_processor
# ========================================
echo ""
echo "========================================"
echo "▶ Step 2/2 : Run L1 processor"
echo "========================================"

ALL_JO_FILES=$(ls "${INPUT_FOLDER}"/JobOrder_*_L1_*.xml 2>/dev/null)
if [ -z "${ALL_JO_FILES}" ]; then
    echo "❌ No L1 JobOrder files found in ${INPUT_FOLDER}"
    exit 1
fi

# Filter if L1_FRAMES_FILTER is set
if [ -z "${L1_FRAMES_FILTER}" ]; then
    echo "ℹ️  No filter set — processing ALL frames"
    JO_L1_FILES="${ALL_JO_FILES}"
else
    echo "ℹ️  Filter set — processing only frames: ${L1_FRAMES_FILTER}"
    JO_L1_FILES=""
    for JO in ${ALL_JO_FILES}; do
        for FRAME_ID in ${L1_FRAMES_FILTER}; do
            if [[ "$(basename ${JO})" == *"_${FRAME_ID}.xml" ]]; then
                JO_L1_FILES="${JO_L1_FILES} ${JO}"
                echo "  ✅ Selected: $(basename ${JO})"
            fi
        done
    done
fi

if [ -z "${JO_L1_FILES}" ]; then
    echo "❌ No L1 JobOrder files to process after filter"
    exit 1
fi

for JO in ${JO_L1_FILES}; do
    echo ""
    echo "--- Running L1 processor on: $(basename ${JO}) ---"
    conda run --name "${BPS_ENV}" bps_l1_processor "${JO}"
    if [ $? -ne 0 ]; then
        echo "❌ L1 processor failed on: ${JO}"
        exit 1
    fi
    echo "✅ Done: $(basename ${JO})"
done

echo ""
echo "✅ L1_chain completed"

✅ BPS environment sourced

▶ Step 1/2 : Create JobOrders L1_chain
Creating JobOrder for L1 processing...
Looking for input files in: /home/jovyan/DATA/01_manaus/Inputs
  ✅ AUX_INS___ found in: /home/jovyan/SCRIPTS/CONFIGURATION_FILE/AUX_441 (1 file(s))
  ✅ AUX_PP1___ found in: /home/jovyan/SCRIPTS/CONFIGURATION_FILE/AUX_441 (1 file(s))
Found 2 RAW__0S file(s): ['BIO_S1_RAW__0S_20251121T095253_20251121T095450_T_G01_M01_C01_T006_F062_01_DNG988', 'BIO_S1_RAW__0S_20251124T095255_20251124T095452_T_G01_M01_C02_T006_F062_01_DK5GET']
Found 2 RAW__0M file(s): ['BIO_S1_RAW__0M_20251124T094512_20251124T095800_T_G01_M01_C02_T006_F____01_DK5M74', 'BIO_S1_RAW__0M_20251121T094510_20251121T095758_T_G01_M01_C01_T006_F____01_DJUQ12']
Found 2 AUX_ATT___ file(s): ['BIO_AUX_ATT____20251124T094456_20251124T095759_01_DIM8IK', 'BIO_AUX_ATT____20251121T094455_20251121T095757_01_DIGNJT']
Found 2 AUX_ORB___ file(s): ['BIO_AUX_ORB____20251121T094455_20251121T095757_01_DIGNJT', 'BIO_AUX_ORB____20251124T094456_2025

2026-04-14T15:06:32.127000 jupyter-biomass-giovanna-palumbo-serco-com L1_P 04.42 L1_P [0000008843]: [W] RAMDISK not specified via Job Order




✅ Done: JobOrder_01_manaus_L1_chain_20251121T09525_V04.42_306.xml

--- Running L1 processor on: JobOrder_01_manaus_L1_chain_20251124T09525_V04.42_306.xml ---
2026-04-14T15:10:17.109000 jupyter-biomass-giovanna-palumbo-serco-com L1_P 04.42 L1_P [0000008910]: [I] Biomass L1 Processor started
2026-04-14T15:10:17.109000 jupyter-biomass-giovanna-palumbo-serco-com L1_P 04.42 L1_P [0000008910]: [I] Job order file: /home/jovyan/DATA/01_manaus/JobOrder_01_manaus_L1_chain_20251124T09525_V04.42_306.xml
2026-04-14T15:10:17.109000 jupyter-biomass-giovanna-palumbo-serco-com L1_P 04.42 L1_P [0000008910]: [I] Using intermediate data directory specified via Job Order
2026-04-14T15:10:17.109000 jupyter-biomass-giovanna-palumbo-serco-com L1_P 04.42 L1_P [0000008910]: [I] Intermediate data directory: /home/jovyan/DATA/01_manaus/intermediates_20251124T095302598698
2026-04-14T15:10:17.109000 jupyter-biomass-giovanna-palumbo-serco-com L1_P 04.42 L1_P [0000008910]: [I] Intermediate data saving enabled
2026-0

2026-04-14T15:10:17.109000 jupyter-biomass-giovanna-palumbo-serco-com L1_P 04.42 L1_P [0000008910]: [W] RAMDISK not specified via Job Order



✅ Done: JobOrder_01_manaus_L1_chain_20251124T09525_V04.42_306.xml

✅ L1_chain completed


## 5. 📚 STA Chain

Generates the STA JobOrders and runs `bps_stack_processor` on each of them.

- **Step 1** — runs `JOBuilder.py STA_chain` (env: `biofetch`) to generate the STA JobOrder XML files
- **Step 2** — runs `bps_stack_processor` (env: `BPS_441`) on each JobOrder

In [6]:
%%bash -s "$JO_ENV" "$JOBUILDER" "$PROCESSOR_VERSION" "$INPUT_FOLDER" "$MISSION_PHASE" "$CONFIGURATION_BPS_DIR" "$BPS_ENV"
JO_ENV=$1
JOBUILDER=$2
PROCESSOR_VERSION=$3
INPUT_FOLDER=$4
MISSION_PHASE=$5
CONFIGURATION_BPS_DIR=$6
BPS_ENV=$7

SCRIPTS_DIR="${CONFIGURATION_BPS_DIR}/tools/scripts"
source "${SCRIPTS_DIR}/set_environment.bash"
echo "✅ BPS environment sourced"

# ========================================
# Step 1/2 : Create JobOrders STA_chain
# ========================================
echo ""
echo "========================================"
echo "▶ Step 1/2 : Create JobOrders STA_chain"
echo "========================================"
conda run --name "${JO_ENV}" python "${JOBUILDER}" STA_chain "${PROCESSOR_VERSION}" "${INPUT_FOLDER}" "${MISSION_PHASE}"
if [ $? -ne 0 ]; then echo "❌ JOBuilder STA_chain failed, stopping"; exit 1; fi

# ========================================
# Step 2/2 : Run bps_stack_processor for each STA JobOrder
# ========================================
echo ""
echo "========================================"
echo "▶ Step 2/2 : Run Stack processor"
echo "========================================"
JO_STA_FILES=$(ls "${INPUT_FOLDER}"/JobOrder_STA_*.xml 2>/dev/null)
if [ -z "${JO_STA_FILES}" ]; then
    echo "❌ No STA JobOrder files found in ${INPUT_FOLDER}"
    exit 1
fi

for JO in ${JO_STA_FILES}; do
    echo ""
    echo "--- Running Stack processor on: $(basename ${JO}) ---"
    conda run --name "${BPS_ENV}" bps_stack_processor "${JO}"
    if [ $? -ne 0 ]; then
        echo "❌ Stack processor failed on: ${JO}"
        exit 1
    fi
    echo "✅ Done: $(basename ${JO})"
done

echo ""
echo "✅ STA_chain completed"

✅ BPS environment sourced

▶ Step 1/2 : Create JobOrders STA_chain

[INFO] Creating STA JobOrders by stack (FRAME-based)
[INFO] Processor version : 04.42
[INFO] Mission phase     : TOMOGRAPHIC
[INFO] Overlap threshold : 0.7
  ✅ AUX_PPS___ found in: /home/jovyan/SCRIPTS/CONFIGURATION_FILE/AUX_441 (1 file(s))
[INFO] AUX_PPS___ file path: /home/jovyan/SCRIPTS/CONFIGURATION_FILE/AUX_441/BIO_AUX_PPS____20250522T000000_20260131T000000_01_DIE6O0
BIO_S1_DGM__1S_20251124T095304_20251124T095325_T_G01_M01_C02_T006_F306_01_DPUYAH
BIO_S1_SCS__1M_20251121T095303_20251121T095323_T_G01_M01_C01_T006_F306_01_DPUY4R
BIO_S1_SCS__1M_20251124T095304_20251124T095325_T_G01_M01_C02_T006_F306_01_DPUYAB
BIO_S1_SCS__1S_20251124T095304_20251124T095325_T_G01_M01_C02_T006_F306_01_DPUY9Q
---------------------------------------------------------
/home/jovyan/DATA/01_manaus/OUTPUT_L1/BIO_S1_SCS__1S_20251124T095304_20251124T095325_T_G01_M01_C02_T006_F306_01_DPUY9Q
__init__
ionosphereCorrection_phaseScreen
ionosphereCorr

2026-04-14T15:15:38.762000 jupyter-biomass-giovanna-palumbo-serco-com STA_P 04.42 STA_P [0000009021]: [W] Got 2 L1a products but 7 or 8 are expected for a TOMOGRAPHIC mission
2026-04-14T15:15:38.763000 jupyter-biomass-giovanna-palumbo-serco-com STA_P 04.42 STA_P [0000009021]: [W] Found L1a products with non-nominal swath duration [s]: {'BIO_S1_SCS__1S_20251121T095303_20251121T095323_T_G01_M01_C01_T006_F306_01_DPUY45': 20.39, 'BIO_S1_SCS__1S_20251124T095304_20251124T095325_T_G01_M01_C02_T006_F306_01_DPUY9Q': 20.39}
2026-04-14T15:15:38.763000 jupyter-biomass-giovanna-palumbo-serco-com STA_P 04.42 STA_P [0000009021]: [W] Found L1a products with nonzero quality index: ['BIO_S1_SCS__1S_20251121T095303_20251121T095323_T_G01_M01_C01_T006_F306_01_DPUY45', 'BIO_S1_SCS__1S_20251124T095304_20251124T095325_T_G01_M01_C02_T006_F306_01_DPUY9Q']



## 6. 🌳 L2A Chain

Generates the L2A JobOrders and runs `bps_l2a_processor` on all frames.

- **Step 1** — runs `JOBuilder.py L2A_chain` (env: `biofetch`) to generate the L2A JobOrder XML files
- **Step 2** — runs `bps_l2a_processor` (env: `BPS_441`) on all frames

In [7]:
%%bash -s "$JO_ENV" "$JOBUILDER" "$PROCESSOR_VERSION" "$INPUT_FOLDER" "$CONFIGURATION_BPS_DIR" "$BPS_ENV"
JO_ENV=$1
JOBUILDER=$2
PROCESSOR_VERSION=$3
INPUT_FOLDER=$4
CONFIGURATION_BPS_DIR=$5
BPS_ENV=$6

SCRIPTS_DIR="${CONFIGURATION_BPS_DIR}/tools/scripts"
source "${SCRIPTS_DIR}/set_environment.bash"
echo "✅ BPS environment sourced"

# ========================================
# Step 1/2 : Create JobOrders L2A_chain
# ========================================
echo ""
echo "========================================"
echo "▶ Step 1/2 : Create JobOrders L2A_chain"
echo "========================================"
conda run --name "${JO_ENV}" python "${JOBUILDER}" L2A_chain "${PROCESSOR_VERSION}" "${INPUT_FOLDER}"
if [ $? -ne 0 ]; then echo "❌ JOBuilder L2A_chain failed, stopping"; exit 1; fi

# ========================================
# Step 2/2 : Run bps_l2a_processor
# ========================================
echo ""
echo "========================================"
echo "▶ Step 2/2 : Run L2A processor"
echo "========================================"

JO_L2A_FILES=$(ls "${INPUT_FOLDER}"/JobOrder_L2A_*.xml 2>/dev/null)
if [ -z "${JO_L2A_FILES}" ]; then
    echo "❌ No L2A JobOrder files found in ${INPUT_FOLDER}"
    exit 1
fi

echo "ℹ️  Processing ALL frames"

for JO in ${JO_L2A_FILES}; do
    echo ""
    echo "--- Running L2A processor on: $(basename ${JO}) ---"
    conda run --name "${BPS_ENV}" bps_l2a_processor "${JO}"
    if [ $? -ne 0 ]; then
        echo "❌ L2A processor failed on: ${JO}"
        exit 1
    fi
    echo "✅ Done: $(basename ${JO})"
done

echo ""
echo "✅ L2A_chain completed"

✅ BPS environment sourced

▶ Step 1/2 : Create JobOrders L2A_chain

[INFO] Creating JobOrders L2A (grouped by FRAME)
[INFO] Root folder: /home/jovyan/DATA/01_manaus
  ✅ AUX_PP2_2A_ found in: /home/jovyan/SCRIPTS/CONFIGURATION_FILE/AUX_441 (1 file(s))
[INFO] PP2 file: /home/jovyan/SCRIPTS/CONFIGURATION_FILE/AUX_441/BIO_AUX_PP2_2A_20170101T000000_20991231T000000_01_BHDHC0

--------------------------------------------------
[INFO] Processing folder: OUTPUT_STA_F306_V04.42

[INFO] FRAME: F306
[INFO] Number of STA__1S in stack: 2
[INFO] Writing JobOrder: /home/jovyan/DATA/01_manaus/JobOrder_L2A_F306_V04.42.xml
Create JobOrder: /home/jovyan/DATA/01_manaus/JobOrder_L2A_F306_V04.42.xml

[OK] All L2A JobOrders created.


▶ Step 2/2 : Run L2A processor
ℹ️  Processing ALL frames

--- Running L2A processor on: JobOrder_L2A_F306_V04.42.xml ---
2026-04-14T15:21:46.289000 jupyter-biomass-giovanna-palumbo-serco-com L2A_P 04.42 L2A_P [0000009234]: [I] Biomass L2a Processor started
2026-04-14T15:21:46.2

2026-04-14T15:21:53.086000 jupyter-biomass-giovanna-palumbo-serco-com L2A_P 04.42 L2A_P [0000009234]: [W] Input L1c acquisition staQuality overallProductQualityIndex=3758096393: l1a source product is invalid
2026-04-14T15:21:58.941000 jupyter-biomass-giovanna-palumbo-serco-com L2A_P 04.42 L2A_P [0000009234]: [W] Input L1c acquisition staQuality overallProductQualityIndex=3758096393: l1a source product is invalid
2026-04-14T15:22:03.412000 jupyter-biomass-giovanna-palumbo-serco-com L2A_P 04.42 L2A_P [0000009234]: [W]     invalid samples before / after Geometry LUT interpolation: 0.990% / 0.990%
2026-04-14T15:22:05.125000 jupyter-biomass-giovanna-palumbo-serco-com L2A_P 04.42 L2A_P [0000009234]: [W]     invalid samples before / after SKP LUT interpolation: 0.990% / 0.990%
2026-04-14T15:22:57.247000 jupyter-biomass-giovanna-palumbo-serco-com L2A_P 04.42 L2A_P [0000009234]: [W]         0.000% of vertical wavenumbers pixels removed for acquisition 1 of 2
2026-04-14T15:22:57.253000 jupyter-b


✅ Done: JobOrder_L2A_F306_V04.42.xml

✅ L2A_chain completed
